In [36]:
import pandas as pd
import requests
from io import StringIO

In [37]:
df_nger = pd.read_csv("NGER.ID0243.csv")

df_accredited = pd.read_csv("power-stations-and-projects-accredited.csv")
df_committed = pd.read_csv("power-stations-and-projects-committed.csv")
df_probable = pd.read_csv("power-stations-and-projects-probable.csv")

In [38]:
abs_url = "https://data.api.abs.gov.au/rest/data/ABS,ABS_REGIONAL_ASGS2021,/..AUS.A?startPeriod=2020&dimensionAtObservation=AllDimensions"

headers = {
    "Accept": "application/vnd.sdmx.data+csv;labels=both"
}

response = requests.get(abs_url, headers=headers, timeout=60)
response.raise_for_status()

df_abs = pd.read_csv(StringIO(response.text))

## merge power-stations-and-projects data

In [39]:
# remove extra spaces in column names
df_accredited.columns = df_accredited.columns.str.strip()
df_committed.columns = df_committed.columns.str.strip()
df_probable.columns = df_probable.columns.str.strip()

# -------------------------
# accredited
# -------------------------
df_accredited_full = df_accredited.rename(columns={
    "Power station name": "project_name",
    "State": "state",
    "Installed capacity (MW)": "capacity_mw",
    "Fuel Source (s)": "fuel_source",
    "Accreditation code": "accreditation_code",
    "Postcode": "postcode",
    "Accreditation start date": "accreditation_start_date",
    "Approval date": "approval_date"
}).copy()

df_accredited_full["committed_date"] = pd.NA
df_accredited_full["project_status"] = "accredited"

# -------------------------
# committed
# -------------------------
df_committed_full = df_committed.rename(columns={
    "Project Name": "project_name",
    "State": "state",
    "MW Capacity": "capacity_mw",
    "Fuel Source": "fuel_source",
    "Committed Date (Month/Year)": "committed_date"
}).copy()

df_committed_full["accreditation_code"] = pd.NA
df_committed_full["postcode"] = pd.NA
df_committed_full["accreditation_start_date"] = pd.NA
df_committed_full["approval_date"] = pd.NA
df_committed_full["project_status"] = "committed"

# -------------------------
# probable
# -------------------------
df_probable["MW Capacity"] = (
    df_probable["MW Capacity"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

df_probable_full = df_probable.rename(columns={
    "Probable Projects": "project_name",
    "State": "state",
    "MW Capacity": "capacity_mw",
    "Fuel Source": "fuel_source"
}).copy()

df_probable_full["capacity_mw"] = pd.to_numeric(df_probable_full["capacity_mw"], errors="coerce")
df_probable_full["accreditation_code"] = pd.NA
df_probable_full["postcode"] = pd.NA
df_probable_full["accreditation_start_date"] = pd.NA
df_probable_full["approval_date"] = pd.NA
df_probable_full["committed_date"] = pd.NA
df_probable_full["project_status"] = "probable"

# -------------------------
# unify column order
# -------------------------
cer_columns = [
    "project_name",
    "state",
    "capacity_mw",
    "fuel_source",
    "project_status",
    "accreditation_code",
    "postcode",
    "accreditation_start_date",
    "approval_date",
    "committed_date"
]

df_accredited_full = df_accredited_full.reindex(columns=cer_columns)
df_committed_full = df_committed_full.reindex(columns=cer_columns)
df_probable_full = df_probable_full.reindex(columns=cer_columns)

# -------------------------
# combine all CER tables
# -------------------------
df_cer = pd.concat(
    [df_accredited_full, df_committed_full, df_probable_full],
    ignore_index=True
)

In [40]:
print("NGER columns:")
display(df_nger.columns)

print("CER columns:")
display(df_cer.columns)


print("ABS columns:")
display(df_abs.columns)

NGER columns:


Index(['Reporting entity', 'Facility name', 'Type', 'State',
       'Electricity production GJ', 'Electricity production MWh',
       'Total scope 1 emissions t CO2 e', 'Total scope 2 emissions t CO2 e',
       'Total emissions t CO2 e', 'Emission intensity t CO2 e MWh',
       'Grid connected', 'Grid', 'Primary fuel', 'Important notes'],
      dtype='object')

CER columns:


Index(['project_name', 'state', 'capacity_mw', 'fuel_source', 'project_status',
       'accreditation_code', 'postcode', 'accreditation_start_date',
       'approval_date', 'committed_date'],
      dtype='object')

ABS columns:


Index(['DATAFLOW', 'MEASURE: Data Item', 'REGIONTYPE: Region Type',
       'ASGS_2021: Region', 'FREQUENCY: Frequency', 'TIME_PERIOD: Time Period',
       'OBS_VALUE', 'UNIT_MEASURE: Unit of Measure',
       'UNIT_MULT: Unit of Multiplier', 'OBS_STATUS: Observation Status',
       'OBS_COMMENT: Observation Comment'],
      dtype='object')